In [2]:
import os
from pymongo import MongoClient
import pandas as pd
from dotenv import load_dotenv

# Загружаем переменные окружения из файла file.env
load_dotenv("file.env")

# Настройки подключения к MongoDB 
def get_connection():
    username = os.getenv("MONGO_INITDB_ROOT_USERNAME")
    password = os.getenv("MONGO_INITDB_ROOT_PASSWORD")
    port = os.getenv("MONGO_PORT")
    host = "localhost"
    db_name = os.getenv("MONGO_DB_NAME")
    uri = f"mongodb://{username}:{password}@{host}:{port}/{db_name}?authSource=admin" #без параметра authSource=admin была ошибка Authentication failed
    return MongoClient(uri)

def get_users_stats (query: dict, projection: dict = None):
    db_name = os.getenv("MONGO_DB_NAME")
    with get_connection() as client:
        db = client[db_name]
        collection = db["users"] 
        data = list(collection.find(query, projection))
        df = pd.DataFrame(data)
        return df

if __name__ == "__main__":
    get_connection()

In [3]:
# Задача 1. Найти пользователей младше 30 лет
query = {"age": {"$lt": 30}}
users_df = get_users_stats(query)
display(users_df)

KeyboardInterrupt: 

Задача 1. Найти пользователей младше 30 лет (запрос на MongoDB Query Language)
db.users.find({age: {
  $lt: 30
}})

In [37]:
#Задача 2 Вывести только имя и email пользователей, которые живут в Brazil 
query = {"country": {"$in": ["Brazil"]}}
fields = {"name": 1, "email": 1, "_id": 0}
users_df = get_users_stats(query, fields)
display(users_df)

,name,email
0,Laura Mora,mcintoshbruce@example.org
1,Mark Baker,davidpetty@example.net
2,Blake Vazquez MD,chelseaberger@example.net
3,Ashley Martinez,andersonbrian@example.org
4,Elizabeth Osborne,jamesbarbara@example.com
5,Gabriel Esparza,allison59@example.org
6,Robert Jones,fjohnson@example.net
7,Linda Santos,xhughes@example.com
8,Joshua Hall,branchcandice@example.net
9,Tara Walters,thomasharris@example.org


Задача 2. Вывести только имя и email пользователей, которые живут в Brazil (запрос на MongoDB Query Language)
db.users.find({country: {
  $in: ['Brazil']
}},
{name: 1, email: 1, _id: 0}   
)

In [ ]:
#Задача 3. Вывести пользователей старше 30 лет и младше 45, отсортированные по возрасту
query = {"age": {"$gt": 30, "$lt": 45}}
fields = {"name": 1, "email": 1, "age": 1, "_id": 0}
users_df = get_users_stats(query, fields)
users_df = users_df.sort_values(by="age", ascending=False)
display(users_df)

,name,email,age
365,Maria Hill,hsmith@example.net,44
327,David Sims,ubarnes@example.net,44
341,Jesse Fernandez,ydavenport@example.org,44
372,Jonathan Blair,brownkimberly@example.net,44
371,Jared Patton,schultzkathy@example.org,44
...,...,...,...
132,Jessica Martin,wallaceelizabeth@example.com,31
131,Carl Espinoza,ramirezeileen@example.com,31
75,Albert Hicks,jason84@example.net,31
67,Sylvia Carrillo,john22@example.org,31


Задача 3. Вывести пользователей старше 30 лет и младше 45, отсортированные по возрасту
db.users.aggregate([
  {$match: {age: {$gt: 30, $lt: 45}
  }},
  {$sort: {age: -1}},
  {
    $project: {
      name: 1,
      age: 1,
      _id: 0
    }
  }
]) 

Задача 4. Посчитать количество пользователей по странам

db.users.aggregate([
    {$group: {
        _id: "$country",
        count: {$sum: 1}
    }}
])

Задача 5. Посчитать, сколько всего активных пользователей в каждой стране.

db.users.aggregate([
    {$match: {is_active: true }},
    {$group: {
        _id: "$country",
        count: {$sum: 1}
    }}
])

Задача 6*. Найти минимальный и максимальный возраст

db.users.aggregate([
    {$group: {
        _id: null,
        min_age: {$min: "$age"},
        max_age: {$max: "$age"}
    }}
])

Задача 7*. Вывести имя и возраст самого старшего пользователя
db.users.aggregate([
    { $sort: { age: -1 } },
    { $limit: 1 },
    { $project: { name: 1, age: 1, _id: 0 } }
])